In [16]:
import cv2
import torch
import numpy as np
import os
from tqdm import tqdm
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F

In [83]:
class Config:
    IMAGE_SIZE = 256
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
class ResBlock(nn.Module):
    """Simple residual block. Skip connection preserves low-level features
    and allows gradients to flow without vanishing through deep encoders."""
    def __init__(self, ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ch, ch, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(ch, ch, 3, padding=1),
        )
    def forward(self, x):
        return F.relu(self.net(x) + x, inplace=True)

class PyramidEncoder(nn.Module):
    """
    3-scale pyramid encoder.
    Returns (e1, e2, e3) where:
      e1: stride-2 downsampled  (H/2, W/2,  ch=32)  ← finest scale
      e2: stride-4 downsampled  (H/4, W/4,  ch=64)
      e3: same spatial as e2    (H/4, W/4,  ch=128)  ← bottleneck
    Each scale has a ResBlock to increase effective receptive field.
    """
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1), nn.ReLU(inplace=True),
            ResBlock(32)
        )
        self.layer2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(inplace=True),
            ResBlock(64)
        )
        self.layer3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(inplace=True),
            ResBlock(128)
        )

    def forward(self, x):
        e1 = self.layer1(x)   
        e2 = self.layer2(e1)  
        e3 = self.layer3(e2)  
        return e1, e2, e3


class FlowEstimator(nn.Module):
    
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(256, 128, 3, padding=1), nn.ReLU(inplace=True),
            ResBlock(128),
            nn.Conv2d(128, 64, 3, padding=1),  nn.ReLU(inplace=True),
            nn.Conv2d(64,  5,  3, padding=1),  # 4 flow channels + 1 occ
        )

    def forward(self, F0, F1):
        return self.net(torch.cat([F0, F1], dim=1))


def warp(img, flow):
    
    B, C, H, W = img.shape
    grid_y, grid_x = torch.meshgrid(
        torch.linspace(-1, 1, H, device=img.device),
        torch.linspace(-1, 1, W, device=img.device),
        indexing='ij'
    )
    base_grid = torch.stack([grid_x, grid_y], dim=-1)          
    base_grid = base_grid.unsqueeze(0).expand(B, -1, -1, -1)   
    scale = torch.tensor([2.0/W, 2.0/H], device=img.device, dtype=img.dtype)
    flow_norm = flow.permute(0, 2, 3, 1) * scale              
    sample_grid = base_grid + flow_norm
    return F.grid_sample(img, sample_grid, mode='bilinear',
                         padding_mode='border', align_corners=True)


class SelfAttention(nn.Module):
    
    def __init__(self, dim, attn_size=16):
        super().__init__()
        self.attn_size = attn_size
        self.q = nn.Conv2d(dim, dim, 1)
        self.k = nn.Conv2d(dim, dim, 1)
        self.v = nn.Conv2d(dim, dim, 1)

    def forward(self, x):
        B, C, H, W = x.shape
        S = self.attn_size
       
        xs = F.adaptive_avg_pool2d(x, (S, S))
        q = self.q(xs).view(B, C, -1).permute(0, 2, 1)
        k = self.k(xs).view(B, C, -1).permute(0, 2, 1)
        v = self.v(xs).view(B, C, -1).permute(0, 2, 1)
        out = F.scaled_dot_product_attention(q, k, v)
        out = out.permute(0, 2, 1).view(B, C, S, S)
        out = F.interpolate(out, size=(H, W), mode='bilinear', align_corners=False)
        return out + x


class SynthesisNet(nn.Module):
    
    def __init__(self):
        super().__init__()
       
        self.fuse_bot = nn.Sequential(
            nn.Conv2d(256, 128, 3, padding=1), nn.ReLU(inplace=True),
            ResBlock(128)
        )

        
        self.up2 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(128 + 64*2, 64, 3, padding=1), nn.ReLU(inplace=True),
            ResBlock(64)
        )
        self.aux_head2 = nn.Conv2d(64, 3, 1)   

        
        self.up1_upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.up1_conv = nn.Sequential(
            nn.Conv2d(64 + 32*2 + 3*2, 32, 3, padding=1), nn.ReLU(inplace=True),
            ResBlock(32)
        )
        self.aux_head1 = nn.Conv2d(32, 3, 1)   


        self.out_head = nn.Sequential(
            nn.Conv2d(32, 16, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(16,  3, 3, padding=1),
            nn.Tanh()
        )

    def forward(self, Ft0, Ft1, e1_0, e1_1, e2_0, e2_1, W0, W1):
        
    
        x = self.fuse_bot(torch.cat([Ft0, Ft1], dim=1)) 

        
        x = self.up2(torch.cat([x, e2_0, e2_1], dim=1))  
        aux2 = self.aux_head2(x)                          

    
        x = self.up1_upsample(x)                                             
        H_full, W_full = x.shape[2], x.shape[3]
        e1_0_up = F.interpolate(e1_0, size=(H_full, W_full),
                                mode='bilinear', align_corners=False)         
        e1_1_up = F.interpolate(e1_1, size=(H_full, W_full),
                                mode='bilinear', align_corners=False)           
        x = self.up1_conv(torch.cat([x, e1_0_up, e1_1_up, W0, W1], dim=1))
        aux1 = self.aux_head1(x)                                            

        out = self.out_head(x)   
        return out, aux1, aux2


In [ ]:
class VFIModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.encoder  = PyramidEncoder()
        self.attn     = SelfAttention(128, attn_size=16)  
        self.flow_est = FlowEstimator()
        self.synthesis = SynthesisNet()

    def forward(self, I0, I1, t):
        
        e1_0, e2_0, e3_0 = self.encoder(I0)  
        e1_1, e2_1, e3_1 = self.encoder(I1)

        F0 = self.attn(e3_0)
        F1 = self.attn(e3_1)

        flow_out = self.flow_est(F0, F1)  
        flow_0t  = flow_out[:, :2]         
        flow_1t  = flow_out[:, 2:4]       
        occ      = torch.sigmoid(flow_out[:, 4:5])  

        t_view = t.view(-1, 1, 1, 1)

     
        H_full, W_full = I0.shape[2], I0.shape[3]
        H_bot = flow_0t.shape[2]
        scale_factor = H_full / H_bot
        flow_0t_full = F.interpolate(flow_0t, scale_factor=scale_factor, mode='bilinear', align_corners=False) * scale_factor
        flow_1t_full = F.interpolate(flow_1t, scale_factor=scale_factor, mode='bilinear', align_corners=False) * scale_factor
        W0_px = warp(I0, flow_0t_full * t_view)        
        W1_px = warp(I1, flow_1t_full * (1 - t_view))  

        
        Ft0 = warp(F0, flow_0t * t_view)
        Ft1 = warp(F1, flow_1t * (1 - t_view))

       
        synth, aux1, aux2 = self.synthesis(Ft0, Ft1, e1_0, e1_1, e2_0, e2_1, W0_px, W1_px)

  
        occ_full = F.interpolate(occ, size=(H_full, W_full), mode='bilinear', align_corners=False)
        base = occ_full * W0_px + (1 - occ_full) * W1_px
        pred = (synth + base).clamp(-1, 1)

        return pred, aux1, aux2

In [86]:
model = VFIModel().to(Config.DEVICE)

checkpoint = torch.load(
    r"D:\Stoarge\VNIT\3rd Year\sem6\study material\H_IACV_4\IACV_project\vfi256_v7_best_model.pth",
    map_location=Config.DEVICE
)

model.load_state_dict(checkpoint["model"])
model.eval()

print("✅ Model loaded")

✅ Model loaded


C:\Users\karti\AppData\Local\Temp\ipykernel_28424\3609987660.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(


In [87]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((Config.IMAGE_SIZE, Config.IMAGE_SIZE)),
    transforms.ToTensor(),
])

In [88]:
def extract_frames(video_path, output_folder="Result_3/temp_frames"):
    os.makedirs(output_folder, exist_ok=True)

    cap = cv2.VideoCapture(video_path)

    frames = []
    idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        path = os.path.join(output_folder, f"{idx:05d}.png")
        cv2.imwrite(path, frame)
        frames.append(path)
        idx += 1

    cap.release()
    print(f"✅ Extracted {len(frames)} frames")
    return frames

In [ ]:
def read_image(path):
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return transform(img)

def tensor_to_image(tensor):
   
    if isinstance(tensor, (tuple, list)):
        tensor = tensor[0]

    if tensor.ndim == 4:
        if tensor.shape[0] != 1:
            raise ValueError(f"Expected batch size 1 for image conversion, got shape {tuple(tensor.shape)}")
        tensor = tensor[0]

    if tensor.ndim != 3:
        raise ValueError(f"Expected CHW tensor, got shape {tuple(tensor.shape)}")

    img = tensor.detach().float().cpu()
    
    if img.min() < 0:
        img = (img + 1.0) / 2.0

    img = img.clamp(0.0, 1.0).permute(1, 2, 0).numpy()
    img = (img * 255.0).round().astype(np.uint8)
    return img

In [ ]:
def interpolate_frames(model, frame_paths, num_intermediate=2, output_folder="Result_3/interp_frames"):
    
    os.makedirs(output_folder, exist_ok=True)
    output_frames = []

    idx = 0

    for i in tqdm(range(len(frame_paths)-1)):

        I0 = read_image(frame_paths[i]).unsqueeze(0).to(Config.DEVICE)
        I1 = read_image(frame_paths[i+1]).unsqueeze(0).to(Config.DEVICE)

        first = tensor_to_image(I0[0])
        cv2.imwrite(f"{output_folder}/{idx:05d}.png", cv2.cvtColor(first, cv2.COLOR_RGB2BGR))
        output_frames.append(f"{output_folder}/{idx:05d}.png")
        idx += 1

        for j in range(1, num_intermediate+1):

            t = j / (num_intermediate + 1)
            t_tensor = torch.tensor([t]).float().to(Config.DEVICE)

            with torch.no_grad():
                pred, _, _ = model(I0, I1, t_tensor)

            img = tensor_to_image(pred)

            path = f"{output_folder}/{idx:05d}.png"
            cv2.imwrite(path, cv2.cvtColor(img, cv2.COLOR_RGB2BGR))

            output_frames.append(path)
            idx += 1

    last = read_image(frame_paths[-1]).unsqueeze(0)
    last_img = tensor_to_image(last[0])

    path = f"{output_folder}/{idx:05d}.png"
    cv2.imwrite(path, cv2.cvtColor(last_img, cv2.COLOR_RGB2BGR))
    output_frames.append(path)

    print("✅ Interpolation done")
    return output_frames

In [91]:
def frames_to_video(frame_paths, output_video="Result_3/output.mp4", fps=30):

    first_frame = cv2.imread(frame_paths[0])
    h, w, _ = first_frame.shape

    out = cv2.VideoWriter(
        output_video,
        cv2.VideoWriter_fourcc(*'mp4v'),
        fps,
        (w, h)
    )

    for path in frame_paths:
        frame = cv2.imread(path)
        out.write(frame)

    out.release()
    print(f"✅ Video saved: {output_video}")

In [ ]:

video_path = r"D:\Stoarge\VNIT\3rd Year\sem6\study material\H_IACV_4\IACV_project\clips\video_000094-Scene-121.mp4"
num_frames_to_add = 2 


frames = extract_frames(video_path)

interp_frames = interpolate_frames(
    model,
    frames,
    num_intermediate=num_frames_to_add
)

frames_to_video(
    interp_frames,
    output_video="Result_3/final_clip_output.mp4",
    fps=30
)

✅ Extracted 74 frames


100%|██████████| 73/73 [00:28<00:00,  2.58it/s]


✅ Interpolation done
✅ Video saved: Result_3/final_clip_output.mp4
